In [7]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [8]:
base_dir = r'/home/tinmar/Desktop/Projects/Datasets/Puzzle/tiles'
input_mask_dir = r'/home/tinmar/Desktop/Projects/Datasets/Puzzle/tiles/masks_inv/'
output_polygon_dir = r'/home/tinmar/Desktop/Projects/Datasets/Puzzle/tiles/polygons_inv/'

is_mask_inv = 'inv' in Path(input_mask_dir).name

if is_mask_inv:
    if 'inv' not in Path(output_polygon_dir).name:
        raise ValueError("Output polygon directory name should indicate inverted masks (e.g., contain 'inv').")
    print("Using inverted mask logic.")



if not os.path.exists(input_mask_dir):
    print(f"Input mask directory does not exist: {input_mask_dir}")
if not os.path.exists(output_polygon_dir):
    os.makedirs(output_polygon_dir)


Using inverted mask logic.


In [9]:
def mask_to_polygons(mask_path, output_path, class_id=0, min_area=200):
    # Load mask as grayscale
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None: return
    
    # Ensure binary mask
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    h, w = binary.shape
    
    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    
    polygons = []
    for cnt in contours:
        # Filter noise
        if cv2.contourArea(cnt) < min_area:
            continue
            
        # Flatten and normalize coordinates
        # Reshape(-1, 2) gives us pairs of [x, y]
        poly = cnt.reshape(-1, 2).astype(float)
        poly[:, 0] /= w  # Normalize x
        poly[:, 1] /= h  # Normalize y
        
        # Flatten to x1, y1, x2, y2...
        flat_poly = poly.flatten().tolist()
        polygons.append(flat_poly)

    # Write to file (one polygon per line)
    with open(output_path, 'w') as f:
        for poly in polygons:
            # Convert float list to space-separated string
            poly_str = " ".join([f"{p:.6f}" for p in poly])
            f.write(f"{class_id} {poly_str}\n")

In [10]:
for mask_file in os.listdir(input_mask_dir):
    if mask_file.endswith(('.png', '.jpg', '.jpeg', '.tif')):
        mask_path = os.path.join(input_mask_dir, mask_file)
        output_path = os.path.join(output_polygon_dir, mask_file[:-4] + '.txt')
        mask_to_polygons(mask_path, output_path, class_id=0, min_area=0)  

In [11]:
def visualize_annotations(image_path, poly_path, alpha=0.5):
    img = cv2.imread(image_path)
    if img is None: return
    h, w = img.shape[:2]
    
    overlay = img.copy()

    try:
        with open(poly_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue # Need at least 2 points (4 coords) + class_id
                
                # Convert normalized strings to pixel integers
                coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
                coords[:, 0] *= w
                coords[:, 1] *= h
                pts = coords.astype(np.int32).reshape((-1, 1, 2))

                # Draw filled semi-transparent polygon
                cv2.fillPoly(overlay, [pts], (0, 0, 255)) # Red Fill
                # Draw solid green outline
                cv2.polylines(img, [pts], True, (0, 255, 0), 2)

        # Blend the fill with the original
        result = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
        
        cv2.imshow("Result", result)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    except Exception as e:
        print(f"Error: {e}")

In [12]:
# test_image_name = '1_0.png'
# test_image = os.path.join(base_dir, 'images', test_image_name)
# test_mask = os.path.join(base_dir, 'masks_inv', test_image_name)
# test_poly = os.path.join(base_dir, 'polygons_inv', test_image_name[:-4] + '.txt')

# visualize_annotations(test_image, test_poly, alpha=0.1)